# Module 1: Why Naive RAG Fails

## 🎯 Learning Objectives

In this hands-on lab, you will:
1. **Build a naive RAG pipeline** - the simplest possible implementation
2. **See it fail** on real documents with tables, figures, and complex layouts
3. **Understand WHY** it fails - so you appreciate the solutions in Modules 2-6

---

## 🗺️ What is RAG? (Retrieval-Augmented Generation)

**RAG** lets an LLM answer questions about YOUR documents (that it wasn't trained on).

**The idea is simple:**
1. User asks a question
2. Find relevant pieces of your document
3. Give those pieces + the question to the LLM
4. LLM generates an answer based on YOUR content

```
┌──────────────────────────────────────────────────────────────────────────┐
│                         RAG PIPELINE                                     │
│                                                                          │
│  📄 Your        🔍 Extract    ✂️ Split into   🧮 Convert to              │
│  Document  ──▶   Text    ──▶   Chunks    ──▶   Numbers    ──▶ 📦 Store  │
│  (PDF)          (words)       (pieces)       (embeddings)      (index)   │
│                                                                          │
│  ════════════════════════════════════════════════════════════════════    │
│                                                                          │
│  ❓ User         🔎 Find        📝 Give to      💬 Generate               │
│  Question  ──▶  Similar   ──▶    LLM     ──▶   Answer                   │
│                 Chunks          (context)                                │
└──────────────────────────────────────────────────────────────────────────┘
```

### The Problem We'll Discover

In this module, we intentionally use **naive approaches** to see how they fail:

| Step | Naive Approach | What Goes Wrong |
|------|----------------|-----------------|
| 🔍 Extract | Just grab raw text | Tables become garbage, figures lost |
| ✂️ Chunk | Cut every 500 characters | Sentences cut in half, context lost |

**Let's build it and watch it break!**

In [ ]:
# Visualize the RAG Pipeline
from IPython.display import display, HTML

# Create a visual diagram of RAG
diagram_html = """
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); 
            padding: 30px; border-radius: 15px; margin: 20px 0; font-family: Arial, sans-serif;">
    
    <h3 style="color: #eee; text-align: center; margin-bottom: 25px;">
        🔄 The RAG Pipeline (Retrieval-Augmented Generation)
    </h3>
    
    <!-- Indexing Phase -->
    <div style="margin-bottom: 20px;">
        <div style="color: #4ecdc4; font-weight: bold; margin-bottom: 10px;">📥 INDEXING PHASE (done once)</div>
        <div style="display: flex; align-items: center; justify-content: space-around; flex-wrap: wrap; gap: 10px;">
            <div style="background: #e74c3c; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                📄<br>Document<br><small>(PDF)</small>
            </div>
            <div style="color: #888; font-size: 24px;">→</div>
            <div style="background: #f39c12; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                🔍<br>Extract<br><small>(text)</small>
            </div>
            <div style="color: #888; font-size: 24px;">→</div>
            <div style="background: #9b59b6; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                ✂️<br>Chunk<br><small>(pieces)</small>
            </div>
            <div style="color: #888; font-size: 24px;">→</div>
            <div style="background: #3498db; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                🧮<br>Embed<br><small>(numbers)</small>
            </div>
            <div style="color: #888; font-size: 24px;">→</div>
            <div style="background: #2ecc71; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                📦<br>Index<br><small>(store)</small>
            </div>
        </div>
    </div>
    
    <!-- Query Phase -->
    <div>
        <div style="color: #ff6b6b; font-weight: bold; margin-bottom: 10px;">🔎 QUERY PHASE (every question)</div>
        <div style="display: flex; align-items: center; justify-content: space-around; flex-wrap: wrap; gap: 10px;">
            <div style="background: #1abc9c; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                ❓<br>Question<br><small>(user)</small>
            </div>
            <div style="color: #888; font-size: 24px;">→</div>
            <div style="background: #3498db; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                🧮<br>Embed<br><small>(numbers)</small>
            </div>
            <div style="color: #888; font-size: 24px;">→</div>
            <div style="background: #e67e22; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                🔎<br>Search<br><small>(find similar)</small>
            </div>
            <div style="color: #888; font-size: 24px;">→</div>
            <div style="background: #8e44ad; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                🤖<br>LLM<br><small>(generate)</small>
            </div>
            <div style="color: #888; font-size: 24px;">→</div>
            <div style="background: #27ae60; color: white; padding: 15px; border-radius: 10px; text-align: center; min-width: 100px;">
                💬<br>Answer<br><small>(response)</small>
            </div>
        </div>
    </div>
    
    <div style="color: #aaa; text-align: center; margin-top: 20px; font-size: 14px;">
        ⚠️ In this module, we'll see how <span style="color: #e74c3c;">Extract</span> and 
        <span style="color: #9b59b6;">Chunk</span> can break the entire pipeline!
    </div>
</div>
"""
display(HTML(diagram_html))
print("👆 This is the RAG pipeline. We'll build each step and see where naive approaches fail.")

---

## Step 0: Setup

First, let's load our environment and verify our Azure services are accessible.

In [ ]:
import os
import sys
from pathlib import Path

# Add the src directory to the path so we can import shared utilities
sys.path.append(str(Path("../../src").resolve()))

from utils import load_env

# Load environment variables from .env file
env = load_env()
print("✅ Environment loaded successfully!")
print(f"   Document Intelligence endpoint: {env.get('AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT', 'NOT SET')[:50]}...")

---

## Step 1: Meet Our Sample Document

We'll use a real document from the **Israel M1 Metro Line** project - specifically Station 36 (שדרות הציונות).

This document contains:
- 📊 **Tables** with station specifications
- 🖼️ **Figures** with maps and diagrams  
- 📝 **Text** in Hebrew and English

Let's load it:

In [ ]:
# Define path to our sample document
DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "metro-s36.pdf"

# Verify the file exists
if PDF_PATH.exists():
    size_mb = PDF_PATH.stat().st_size / (1024 * 1024)
    print(f"✅ Found: {PDF_PATH.name}")
    print(f"   Size: {size_mb:.1f} MB")
else:
    print(f"❌ File not found: {PDF_PATH}")
    print("   Please ensure the sample PDFs are in the data/sample-pdfs folder.")

### 👀 Let's See What's in This Document

Here's a preview of the first page - notice the **complex layout** with text, tables, and diagrams:

In [ ]:
from IPython.display import Image, display

# Display the first page of the document
print("📄 First page of metro-s36.pdf:")
print("   Notice the complex layout with text, maps, and structured information.\n")

try:
    display(Image(filename="../../data/module1-images/metro_s36_page1.png", width=700))
except:
    print("   (Image not found - run the image extraction script first)")

---

## Step 2: Naive Text Extraction

### 🗺️ Pipeline Position
```
📄 Document → 🔍 EXTRACT ← WE ARE HERE (doing it WRONG)
```

### What We're Doing (The WRONG Way)

We'll use **Azure AI Document Intelligence** to extract text, but we'll throw away all the structure it gives us. We'll just grab the raw text as one long string.

This simulates what happens when you use simple tools like `pypdf` or `pdfplumber` that don't understand document structure.

### 📚 Azure AI Document Intelligence

Document Intelligence is a powerful service that can:
- Extract text with **reading order** preserved
- Identify **tables** with rows and columns
- Detect **figures** with bounding boxes
- Understand **document structure** (headings, paragraphs)

**But in this naive approach, we'll ignore all that structure!**

> 📖 **Documentation**: [Azure AI Document Intelligence](https://learn.microsoft.com/azure/ai-services/document-intelligence/)

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential

# Initialize the Document Intelligence client
endpoint = env["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]

# Try Entra ID authentication first (recommended), fall back to API key
try:
    print("🔐 Authenticating with Entra ID (DefaultAzureCredential)...")
    credential = DefaultAzureCredential()
    client = DocumentIntelligenceClient(endpoint=endpoint, credential=credential)
    print("✅ Connected with Entra ID")
except Exception as e:
    print(f"⚠️ Entra ID failed, using API key: {e}")
    key = env["AZURE_DOCUMENT_INTELLIGENCE_KEY"]
    client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))
    print("✅ Connected with API key")

In [ ]:
# Extract text from the PDF using Document Intelligence
# We use the 'prebuilt-layout' model which extracts text, tables, and figures

print(f"📄 Analyzing {PDF_PATH.name}...")
print("   This may take 30-60 seconds...\n")

with open(PDF_PATH, "rb") as f:
    poller = client.begin_analyze_document(
        "prebuilt-layout",  # The model that extracts layout structure
        f,
        content_type="application/pdf"
    )
    result = poller.result()

print(f"✅ Analysis complete!")
print(f"   Pages extracted: {len(result.pages)}")
print(f"   Tables found: {len(result.tables) if result.tables else 0}")
print(f"   Figures found: {len(result.figures) if result.figures else 0}")

### ⚠️ The Naive Mistake: Ignoring Structure

Document Intelligence gave us **rich structured data** (tables, figures, paragraphs).

But in naive RAG, we **throw it all away** and just grab the raw text:

In [ ]:
# THE NAIVE APPROACH: Just get all content as one big string
# This is what most simple RAG tutorials do!

full_text = result.content  # Just the raw text, no structure

print(f"📝 Extracted {len(full_text):,} characters of raw text")
print(f"\n--- First 500 characters ---\n")
print(full_text[:500])
print("\n...")

### 🔍 Look at the Output Above - What's Wrong?

The extracted text above **looks reasonable** at first glance:
- ✅ We can see the station name: "תחנה מס׳ 36 | שדרות הציונות"
- ✅ We can see the location description
- ✅ We can read about the neighborhood (שכונת נחלת יהודה)

**But look closer - what's MISSING?**

| What the PDF Has | What We Got |
|------------------|-------------|
| 📊 **Tables** with station specs (dimensions, capacity, etc.) | ❌ Table data is **flattened** into messy text - column relationships are GONE |
| 🖼️ **Maps and diagrams** showing station layout | ❌ **Completely missing** - images aren't in the text at all! |
| 📐 **Structured sections** with clear headers | ⚠️ Headers are mixed with body text - no hierarchy |
| 🔢 **Numbered lists** and bullet points | ⚠️ Formatting lost - just plain text |

**The text LOOKS complete, but it's actually broken for Q&A.**

Try asking an LLM: *"What are the dimensions of the station platform?"* - the answer is in a TABLE that we just destroyed!

---

## Step 3: Naive Chunking

### 🗺️ Pipeline Position
```
📄 Document → 🔍 Extract → ✂️ CHUNK ← WE ARE HERE (doing it WRONG)
```

### What We're Doing (The WRONG Way)

We'll split the text into fixed-size chunks of 500 characters with 50-character overlap.

This is **blind to document structure** - it will cut through:
- Middle of sentences
- Middle of tables
- Separate headers from their content

In [ ]:
def naive_chunk_text(text, chunk_size=500, overlap=50):
    """
    NAIVE chunking: Split text into fixed-size pieces.
    
    This is WRONG because it:
    - Cuts through sentences
    - Splits tables in half
    - Separates context from content
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)  # Move forward with overlap
    return chunks

# Create naive chunks
chunks = naive_chunk_text(full_text)

print(f"✂️ Created {len(chunks)} chunks")
print(f"   Chunk size: 500 characters")
print(f"   Overlap: 50 characters")

### 💥 Failure Mode 1: Context Gets Split

Let's look at a chunk from the middle of the document:

In [ ]:
# Look at a chunk from the middle of the document
sample_idx = len(chunks) // 2
sample_chunk = chunks[sample_idx]

print(f"📄 Chunk #{sample_idx} (from middle of document):")
print("=" * 60)
print(sample_chunk)
print("=" * 60)
print("\n❓ Questions to consider:")
print("   - Does this chunk start at a logical point?")
print("   - Does it end at a logical point?")
print("   - What section of the document is this from?")
print("   - Could an LLM understand the full context?")

### 🔍 Look at the Output Above - What Went Wrong?

**This is a disaster!** Look at what ended up in this chunk:

| What You See | What It Actually Was |
|--------------|---------------------|
| `122 123 102 124 103 152 75...` | 📍 **Map coordinates or parcel numbers** - completely meaningless without the map! |
| `1:1250` | 📐 **Map scale** - but which map? It's orphaned from context |
| `מגורים א׳, מגורים ב׳, מגורים ג׳` | 🏠 **Zoning legend** (Residential A, B, C) - but where's the map it refers to? |
| `תעשייה, מסחר ותעשייה` | 🏭 **Land use categories** - floating without their colored regions |
| `מנספלד- קהת אדריכלים בע״מ` | ✍️ **Architect firm name** from footer - noise mixed with content! |

**The chunk is USELESS because:**

1. ❌ **Starts mid-data** - random numbers with no context
2. ❌ **Ends mid-sentence** - "תחנה מ" is cut off!  
3. ❌ **Mixed content types** - map labels + legend + footer + coordinates all mashed together
4. ❌ **No visual reference** - the MAP that gives meaning to all this is GONE

**Try asking an LLM:** *"What is the zoning around Station 36?"*  
The answer requires the MAP + LEGEND together. This chunk has legend fragments but no map!

---

## 💥 Failure Mode 2: Tables Become Garbage

### The Problem Visualized

Let's look at the TOC (Table of Contents) that spans 2 pages. This is a perfect example of how table structure is destroyed by naive extraction.

**Page 1 of the TOC:**

In [ ]:
from IPython.display import Image, display

print("📊 TOC Page 1 - What the HUMAN sees:")
print("   Notice the clear table structure with Station Name → Page Number mapping\n")

try:
    display(Image(filename="../../data/module1-images/toc_page1.png", width=600))
except:
    print("   (Image not found)")

**Page 2 of the TOC:**

In [ ]:
print("📊 TOC Page 2 - The table continues:")
print("   The table header is on Page 1, but more rows are on Page 2\n")

try:
    display(Image(filename="../../data/module1-images/toc_page2.png", width=600))
except:
    print("   (Image not found)")

### 🔍 Look at the TOC Above - What Gets Destroyed?

**The human sees ONE continuous table spanning 2 pages:**
- `תחנה 35 - קפלן` → Page `1`
- `תחנה 36 - שדרות הציונות` → Page `10`  
- `תחנה 37 - יוסף בורג` → Page `19`
- `תחנה 38 - הרצוג` → Page `28`

Each station has **sections** (אפיון תחנה, סביבת התחנה, תכניות בהכנה, etc.) with their page numbers aligned in columns.

**But Document Intelligence sees TWO separate tables!**

| Page | What DI Extracted | The Problem |
|------|-------------------|-------------|
| Page 1 | Table with headers + first rows | ✅ Has column headers |
| Page 2 | Table with remaining rows | ❌ **NO HEADERS** - orphaned data! |

**When extracted as plain text, you get garbage like:**

```
תחנה 35 - קפלן 1 אפיון תחנה 1 סביבת התחנה ומיפוי אגן היקוות 2 
תכניות בהכנה 3 ייעודי קרקע 4 פיתוח וקישוריות 5 תכניות מאושרות 6 
חתכים 8 מסקנות והמלצות 9 תחנה 36 - שדרות הציונות 10 אפיון תחנה 10...
```

| What Was Lost | Impact |
|---------------|--------|
| **Table continuity** | DI splits into 2 tables - Page 2 has NO column headers! |
| **Column alignment** | Page numbers `1, 2, 3, 4...` look like random numbers in a stream |
| **Row boundaries** | Station 35's sections blend into Station 36's header |
| **Visual hierarchy** | Can't tell station names from section names |

**Try asking an LLM:** *"What page has the land use (ייעודי קרקע) section for Station 36?"*

The answer is `13` - but in flat text, `13` is just another number floating in a sea of Hebrew words!

In [ ]:
# Let's look at how Document Intelligence actually extracted the tables
# (which we're IGNORING in naive RAG)

if result.tables:
    print(f"📊 Document Intelligence found {len(result.tables)} tables")
    print("\n🚨 WAIT - Why 2 tables? The TOC is ONE table spanning 2 pages!")
    print("   Document Intelligence treats each PAGE's portion as a SEPARATE table.")
    print("   This is ANOTHER failure mode - the connection between pages is LOST!\n")
    
    for i, table in enumerate(result.tables):
        page_num = table.bounding_regions[0].page_number if table.bounding_regions else "?"
        print(f"   Table {i+1}: {table.row_count} rows x {table.column_count} columns (Page {page_num})")
    
    print("\n💥 CRITICAL INSIGHT:")
    print("   - Page 1 table has the HEADERS (תחנה, אפיון תחנה, סביבת התחנה...)")
    print("   - Page 2 table has DATA ROWS but NO HEADERS!")
    print("   - In naive RAG, Page 2's data is ORPHANED - no column context!")
else:
    print("No tables found in this document.")

---

## 💥 Failure Mode 3: Figures Are Lost Completely

### The Problem

Documents contain critical visual information:
- Maps and diagrams
- Architectural drawings
- Charts and graphs

**Naive text extraction completely ignores these!**

Let's see an example from our Metro document:

In [ ]:
print("🖼️ Page with diagrams and maps:")
print("   This page contains visual information that naive extraction LOSES\n")

try:
    display(Image(filename="../../data/module1-images/metro_s36_page1.png", width=700))
except:
    print("   (Image not found)")

In [ ]:
# Check what figures Document Intelligence found (which we're ignoring!)

if result.figures:
    print(f"🖼️ Document Intelligence found {len(result.figures)} figures")
    print("\n   These figures have bounding boxes - we could crop and describe them.")
    print("   But in NAIVE RAG, we ignore them completely!\n")
    
    for i, fig in enumerate(result.figures[:4]):  # Show first 4
        print(f"   Figure {i+1}: Page {fig.bounding_regions[0].page_number if fig.bounding_regions else '?'}")
else:
    print("🖼️ No figures detected (or the model didn't extract them)")
    print("   Visual information is LOST in naive RAG!")

---

## Step 4: Create Embeddings

### 🗺️ Pipeline Position
```
📄 Document → 🔍 Extract → ✂️ Chunk → 🧮 EMBED ← WE ARE HERE
```

### 🧠 What Are Embeddings? (Beginner Explanation)

**Embeddings** convert text into numbers (vectors) that capture **meaning**.

Think of it like GPS coordinates for ideas:
- "dog" → `[0.2, 0.8, 0.1, ...]` (1536 numbers)
- "puppy" → `[0.21, 0.79, 0.12, ...]` (similar numbers because similar meaning!)
- "car" → `[0.9, 0.1, 0.7, ...]` (very different numbers)

**Why do we need this?**
- Computers can't understand Hebrew or English directly
- But they CAN compare numbers!
- Similar meanings → Similar numbers → We can find "related" text

```
┌─────────────────────────────────────────────────────────────┐
│                    EMBEDDING SPACE                          │
│                                                             │
│     "כלב" (dog) ●────● "גור" (puppy)                       │
│                  ↖                                          │
│                   close together = similar meaning          │
│                                                             │
│                                    ● "מכונית" (car)         │
│                                      far away = different   │
└─────────────────────────────────────────────────────────────┘
```

> 📖 **Documentation**: [Azure OpenAI Embeddings](https://learn.microsoft.com/azure/ai-services/openai/concepts/understand-embeddings)

In [ ]:
# 👀 Visualize how embeddings work (run this cell to see the diagram)
from IPython.display import display, HTML

embedding_diagram = """
<div style="background: linear-gradient(135deg, #0f0f23 0%, #1a1a3e 100%); 
            padding: 25px; border-radius: 15px; margin: 20px 0; font-family: Arial, sans-serif;">
    
    <h3 style="color: #eee; text-align: center; margin-bottom: 20px;">
        🧮 How Embeddings Capture Meaning
    </h3>
    
    <div style="display: flex; justify-content: space-around; align-items: flex-start; flex-wrap: wrap; gap: 20px;">
        
        <!-- Text to Numbers -->
        <div style="flex: 1; min-width: 280px;">
            <div style="color: #4ecdc4; font-weight: bold; margin-bottom: 10px;">Step 1: Text → Numbers</div>
            <div style="background: #2d2d44; padding: 15px; border-radius: 10px; font-family: monospace; font-size: 12px;">
                <div style="color: #ffd93d;">"כלב" (dog)</div>
                <div style="color: #888;">↓</div>
                <div style="color: #6bcf6b;">[0.23, 0.81, 0.12, 0.45, ... 1536 numbers]</div>
                <br>
                <div style="color: #ffd93d;">"גור כלבים" (puppy)</div>
                <div style="color: #888;">↓</div>
                <div style="color: #6bcf6b;">[0.25, 0.79, 0.14, 0.43, ... similar!]</div>
                <br>
                <div style="color: #ffd93d;">"מכונית" (car)</div>
                <div style="color: #888;">↓</div>
                <div style="color: #ff6b6b;">[0.91, 0.08, 0.76, 0.22, ... different!]</div>
            </div>
        </div>
        
        <!-- Visual Space -->
        <div style="flex: 1; min-width: 280px;">
            <div style="color: #ff6b6b; font-weight: bold; margin-bottom: 10px;">Step 2: Meaning = Position</div>
            <div style="background: #2d2d44; padding: 20px; border-radius: 10px; position: relative; height: 150px;">
                <div style="position: absolute; top: 20px; left: 30px; color: #6bcf6b;">● כלב (dog)</div>
                <div style="position: absolute; top: 35px; left: 80px; color: #6bcf6b;">● גור (puppy)</div>
                <div style="position: absolute; top: 25px; left: 50px; color: #888; font-size: 10px;">close = similar</div>
                <div style="position: absolute; bottom: 30px; right: 40px; color: #ff6b6b;">● מכונית (car)</div>
                <div style="position: absolute; bottom: 50px; right: 30px; color: #888; font-size: 10px;">far = different</div>
            </div>
        </div>
    </div>
    
    <div style="color: #aaa; text-align: center; margin-top: 15px; font-size: 13px;">
        💡 <strong>Key insight</strong>: When you search "מה זה גור?" (what is a puppy?), 
        vector search finds chunks about dogs too - because they're CLOSE in meaning!
    </div>
</div>
"""
display(HTML(embedding_diagram))

In [ ]:
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import numpy as np

# Initialize Azure OpenAI client with Entra ID
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), 
    "https://cognitiveservices.azure.com/.default"
)

openai_client = AzureOpenAI(
    azure_endpoint=env["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=token_provider,
    api_version="2024-02-01"
)

print("✅ Connected to Azure OpenAI")

In [ ]:
# Get the embedding model name from environment
embedding_model = env.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")

def get_embedding(text):
    """Get embedding vector for a text string."""
    response = openai_client.embeddings.create(
        input=[text],
        model=embedding_model
    )
    return response.data[0].embedding

# Embed a subset of chunks (to save time and cost)
# In production, you'd embed all chunks
max_chunks = min(50, len(chunks))
print(f"🧮 Creating embeddings for {max_chunks} chunks...")

embeddings = []
for i, chunk in enumerate(chunks[:max_chunks]):
    if i % 10 == 0:
        print(f"   Processing chunk {i+1}/{max_chunks}...")
    embeddings.append(get_embedding(chunk))

print(f"\n✅ Created {len(embeddings)} embeddings")
print(f"   Each embedding has {len(embeddings[0])} dimensions")

---

## Step 5: Simple Vector Search

### 🗺️ Pipeline Position
```
📄 Document → 🔍 Extract → ✂️ Chunk → 🧮 Embed → 📦 Index → 🔎 SEARCH ← WE ARE HERE
```

### 🔍 What Is Vector Search? (Beginner Explanation)

**Vector Search** finds text chunks that are **semantically similar** to your question.

**How it works:**
1. Your question → converted to embedding (numbers)
2. Compare your question's numbers to ALL chunk numbers
3. Return chunks with the most similar numbers

```
┌─────────────────────────────────────────────────────────────────┐
│                     VECTOR SEARCH                               │
│                                                                 │
│   Your Question: "What is the station type?"                    │
│         ↓                                                       │
│   Question Embedding: [0.3, 0.7, 0.2, ...]                      │
│         ↓                                                       │
│   Compare to ALL chunks:                                        │
│                                                                 │
│   Chunk 1: [0.31, 0.68, 0.19, ...] → 95% similar ✓ MATCH!      │
│   Chunk 2: [0.1, 0.2, 0.9, ...]   → 23% similar                │
│   Chunk 3: [0.8, 0.1, 0.3, ...]   → 41% similar                │
│   Chunk 4: [0.29, 0.71, 0.21, ...] → 92% similar ✓ MATCH!      │
│         ↓                                                       │
│   Return top 3 most similar chunks to the LLM                   │
└─────────────────────────────────────────────────────────────────┘
```

**The magic**: You can ask in Hebrew and find English text (or vice versa) because embeddings capture **meaning**, not exact words!

**The problem**: If the chunk is broken/incomplete, even a "similar" chunk won't have the answer.

In [ ]:
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors."""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def naive_search(query, top_k=3):
    """Search for most relevant chunks using cosine similarity."""
    # Get query embedding
    query_embedding = get_embedding(query)
    
    # Calculate similarity with all chunks
    similarities = []
    for i, chunk_embedding in enumerate(embeddings):
        sim = cosine_similarity(query_embedding, chunk_embedding)
        similarities.append((i, sim, chunks[i]))
    
    # Sort by similarity and return top_k
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

print("✅ Search function ready")

---

## Step 6: Test the Naive RAG - Watch It Fail!

Now let's ask questions that **require the structure we destroyed**:

1. **Table questions** - need column/row relationships
2. **Figure questions** - need visual information  
3. **Cross-reference questions** - need context from multiple places

### Test Question 1: Question Requiring TABLE Structure

In [ ]:
# Test question that REQUIRES TABLE STRUCTURE to answer
# This question needs to find the page number for a specific section - 
# which is stored in a TABLE with columns!

question1 = "באיזה עמוד נמצא פרק ייעודי קרקע של תחנה 36?"  # "What page is the land use section for Station 36?"

print(f"❓ Question: {question1}")
print("   (What page has the 'land use' section for Station 36?)\n")
print("📋 The CORRECT answer is: Page 13 (from the TOC table)")
print("   But can naive RAG find it?\n")
print("🔎 Top 3 retrieved chunks:\n")

results = naive_search(question1, top_k=3)

for i, (chunk_idx, score, chunk_text) in enumerate(results):
    print(f"--- Result {i+1} (similarity: {score:.3f}) ---")
    print(chunk_text[:400] + "..." if len(chunk_text) > 400 else chunk_text)
    print()

print("\n💥 FAILURE ANALYSIS:")
print("   - The answer '13' exists somewhere in the TOC, but WHERE?")
print("   - Is '13' associated with 'ייעודי קרקע' and 'תחנה 36'?")
print("   - Or is '13' just a random number floating in a sea of text?")
print("   - The TABLE STRUCTURE that links section→page is DESTROYED!")

### Test Question 2: Question Requiring FIGURE/MAP Information

In [ ]:
# Test question that REQUIRES VISUAL/MAP information
# This question needs the actual MAP to show zoning colors and boundaries

question2 = "מה הייעוד של החלקה ממזרח לתחנה 36?"  # "What is the zoning east of Station 36?"

print(f"❓ Question: {question2}")
print("   (What is the land use zoning EAST of Station 36?)\n")
print("📋 The CORRECT answer requires looking at the ZONING MAP")
print("   The map shows: תעסוקה (employment/industrial) to the east")
print("   But can naive RAG find it without the map?\n")
print("🔎 Top 3 retrieved chunks:\n")

results = naive_search(question2, top_k=3)

for i, (chunk_idx, score, chunk_text) in enumerate(results):
    print(f"--- Result {i+1} (similarity: {score:.3f}) ---")
    print(chunk_text[:400] + "..." if len(chunk_text) > 400 else chunk_text)
    print()

print("\n💥 FAILURE ANALYSIS:")
print("   - You might see words like 'מגורים', 'תעסוקה', 'מסחר' in the chunks")
print("   - But these are just LEGEND LABELS - not spatial information!")
print("   - The MAP that shows WHERE each zone IS located is COMPLETELY LOST")
print("   - An LLM cannot determine what's 'east' vs 'west' from flat text!")

### Test Question 3: The "Sentence Cut in Half" Problem

This is the simplest failure mode to understand:

**Imagine this sentence in the original document:**
```
סוג התחנה הוא תחנה רגילה והשכונה הסמוכה היא נחלת יהודה.
(The station type is regular station and the nearby neighborhood is Nachalat Yehuda.)
```

**What naive 500-character chunking might do:**
```
┌─────────────────────────────────────────────────┐
│ CHUNK 47 (ends at character 500)                │
│                                                 │
│ ...בלה בלה בלה סוג התחנה הוא תחנה              │
│                              ↑                  │
│                         CUT HERE!               │
└─────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────┐
│ CHUNK 48 (starts at character 450)              │
│                                                 │
│ רגילה והשכונה הסמוכה היא נחלת יהודה...        │
│                                                 │
└─────────────────────────────────────────────────┘
```

**The disaster:**
- Chunk 47 has "סוג התחנה הוא תחנה" (station type is...) but NOT the answer!
- Chunk 48 has "רגילה" (regular) but no context about WHAT is regular!
- Vector search might return Chunk 47 (matches "station type") but the LLM sees incomplete info

In [ ]:
# Let's PROVE this happens in our actual chunks!
# We'll search for a chunk that starts or ends mid-sentence

print("🔍 Let's find chunks that were CUT in the middle of content:\n")

# Find chunks that end with incomplete words/sentences
cut_examples = []
for i, chunk in enumerate(chunks[:50]):  # Check first 50 chunks
    # Check if chunk ends mid-word (no space/punctuation at end)
    if len(chunk) > 10:
        last_char = chunk[-1]
        # If it doesn't end with punctuation or space, it was cut mid-word!
        if last_char not in ' .!?:,،\n':
            cut_examples.append((i, chunk[-100:]))  # Save last 100 chars

print(f"Found {len(cut_examples)} chunks that were CUT mid-content!\n")

if cut_examples:
    # Show first example
    idx, ending = cut_examples[0]
    print(f"📄 Example: Chunk #{idx} ENDS like this:")
    print("=" * 50)
    print(f"...{ending}")
    print("=" * 50)
    print(f"\n⚠️  See how it ends abruptly? The next word is in Chunk #{idx+1}!")
    print("   An LLM receiving this chunk has INCOMPLETE information.")

# Now show the actual search problem
print("\n" + "="*60)
print("📋 Now let's search for 'station type' and see what happens:")
print("="*60 + "\n")

question3 = "מה סוג התחנה?"  # "What is the station type?"

print(f"❓ Question: {question3}")
print("   (What is the station type?)\n")

results = naive_search(question3, top_k=3)

for i, (chunk_idx, score, chunk_text) in enumerate(results):
    print(f"--- Result {i+1} (Chunk #{chunk_idx}, similarity: {score:.3f}) ---")
    # Show shorter preview
    preview = chunk_text[:300].replace('\n', ' ')
    print(preview + "...")
    
    # Check if the actual answer is in this chunk
    has_answer = "תחנה רגילה" in chunk_text
    print(f"\n   ✓ Contains the actual answer 'תחנה רגילה'? {'YES ✅' if has_answer else 'NO ❌'}")
    print()

print("💥 KEY INSIGHT:")
print("   Even if we FIND a relevant chunk, it might not contain the COMPLETE answer!")
print("   The answer could be split across multiple chunks.")

---

## 📊 Summary: Why Naive RAG Fails

| Failure Mode | What Happened | Impact |
|--------------|---------------|--------|
| **Tables destroyed** | Rows/columns become flat text | Can't answer questions about tabular data |
| **Figures lost** | Images completely ignored | Visual information unavailable |
| **Context split** | Fixed-size chunking cuts through sentences | Incomplete or confusing retrieval |
| **Structure ignored** | Headers mixed with content | Hard to find specific sections |

### 🎯 Key Takeaways

1. **Simple text extraction loses structure** - Tables, figures, and layout are critical
2. **Fixed-size chunking is blind** - It doesn't understand document semantics
3. **The RAG pipeline is only as good as its weakest step** - Garbage in = garbage out

---

## ➡️ What's Next?

In the following modules, you'll learn how to fix these problems:

| Module | What You'll Learn | Fixes |
|--------|-------------------|-------|
| **Module 2** | Document Intelligence (properly) | Preserve tables, detect figures |
| **Module 3** | Content Understanding | AI-powered semantic extraction |
| **Module 4** | Smart Chunking | Structure-aware splitting |
| **Module 5** | Search & Retrieval | Hybrid search, semantic ranking |

**Next**: [Module 2 - Document Intelligence](../module-2-doc-intelligence/README.md)

# Module 1: The Problem with Naive RAG

In this module, we will explore why simply "chunking and embedding" technical documents fails. We will build a naive RAG pipeline and observe where it breaks.

In [ ]:
import os
import sys
from pathlib import Path

# Add the src directory to the path so we can import shared utilities
sys.path.append(str(Path("../../src").resolve()))

from utils import load_env
import requests

# Load environment variables
env = load_env()
print("Environment loaded.")

In [ ]:
# Define constants
DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "Basic Electrical Engineering R-20.pdf"

# Verify file exists
if not PDF_PATH.exists():
    print(f"⚠️ Warning: File not found at {PDF_PATH}. Please ensure it is in the data folder.")
else:
    print(f"File found: {PDF_PATH}")

## Step 1: Naive Ingestion

We will use **Azure AI Document Intelligence** to extract text from the PDF.
However, for this "naive" example, we will treat the document as a **single long string of text**, ignoring the rich structure (tables, headers) that the service provides. This simulates what happens when using simple text extractors like `pypdf`.

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential

# Robustness check: Ensure env is loaded even if previous cells were skipped
try:
    env
except NameError:
    import sys
    from pathlib import Path
    # Add src to path if validation fails
    sys.path.append(str(Path("../../src").resolve()))
    from utils import load_env
    env = load_env()
    print("Environment loaded dynamically.")

# Initialize the client
endpoint = env["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]

# Try to use Entra ID (RBAC) first, as Keys might be disabled
try:
    print("Attempting to use DefaultAzureCredential (Entra ID)...")
    credential = DefaultAzureCredential()
    client = DocumentIntelligenceClient(endpoint=endpoint, credential=credential)
except Exception as e:
    print(f"Entra ID init failed, falling back to Key: {e}")
    key = env["AZURE_DOCUMENT_INTELLIGENCE_KEY"]
    client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

# Analyze the document
print(f"Analyzing {PDF_PATH}...")
with open(PDF_PATH, "rb") as f:
    # Pass the file stream as the positional argument (body)
    poller = client.begin_analyze_document(
        "prebuilt-layout", 
        f,
        content_type="application/pdf"
    )
    result = poller.result()

print(f"Analysis complete. Extracted {len(result.pages)} pages.")

# NAIVE APPROACH: Just get all the content as one big string
full_text = result.content
print(f"Total characters: {len(full_text)}")
print("First 500 characters:\n")
print(full_text[:500])

In [ ]:
import azure.ai.documentintelligence
import inspect
print(f"Installed version: {azure.ai.documentintelligence.__version__}")
from azure.ai.documentintelligence import DocumentIntelligenceClient
print(f"Signature: {inspect.signature(DocumentIntelligenceClient.begin_analyze_document)}")

## Step 2: Naive Chunking

Now we will split this text into "chunks" of 500 characters with some overlap. This is a common starting point for RAG, but it is blind to document structure.

In [ ]:
def chunk_text(text, size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += (size - overlap)
    return chunks

chunks = chunk_text(full_text)
print(f"Created {len(chunks)} chunks.")

print("\n--- 🔍 EVIDENCE OF FAILURE 1: LOSS OF CONTEXT ---")
print("Here is a random chunk. notice how it likely starts or ends in the middle of a sentence.")
print("This makes it hard for the LLM to understand what this text is actually about.\n")

import random
# Pick a chunk from the middle of the document where tech details are
if len(chunks) > 50:
    sample_chunk = chunks[50] 
    print(f"--- Chunk #50 ---\n{sample_chunk}\n-----------------")
else:
    print(random.choice(chunks))

In [ ]:
# --- DEEP DIVE: PAGE 8 ANALYSIS ---
from IPython.display import Image, display

print("--- 1. THE VISUAL REALITY (What the human sees) ---")
print("Notice the equation 'I = dQ/dt' and the variable definitions below it.")
print("Notice the footer at the bottom.")
# Display the visual reference (ensure page8.png is in the same folder)
try:
    display(Image(filename="page8.png", width=600))
except:
    print("Image not found. Please verify page8.png exists.")

print("\n--- 2. THE NAIVE CHUNKING REALITY (What the LLM sees) ---")

try:
    # Find Page 8
    page_8 = next(p for p in result.pages if p.page_number == 8)
    
    # Extract text
    page_8_text = ""
    for span in page_8.spans:
        page_8_text += result.content[span.offset : span.offset + span.length]

    # Chunk it
    p8_chunks = chunk_text(page_8_text)

    for i, c in enumerate(p8_chunks):
        print(f"--- Chunk {i+1} ---")
        # PRINT FIX: We use print(c) instead of repr(c) to let newlines render naturally for readability
        print(c)
        print("-------------------")
        
        # Cleanup string for detection (remove newlines/multi-spaces to ensure matching works)
        clean_c = " ".join(c.split())
        
        # 1. Check for the Equation Part (Chunk N)
        if "dQ" in c and "unit is second" not in c:
             print("   ⚠️  WARNING: Equation FOUND, but 't' definition might be missing in this chunk.")

        # 2. Check for the Definition Part (Chunk N+1)
        # We look for 'unit is second' which refers to 't'
        if ("unit is second" in c or "t is the time" in c) and "dQ" not in c:
             print("   🚨 CRITICAL FAILURE (Context Split):")
             print("       This chunk defines 't' ('unit is second'), but the equation 'I=dQ/dt' is MISSING.")
             print("       The equation was likely left behind in the previous chunk. The LLM has lost the connection.")
        
        # 3. Check for Footer noise
        if "MRCET" in c or "EAMCET" in c:
             print("   🚨 NOISE POLLUTION: Footer usage rights/codes merged with physics content.")
        
        print("")

except StopIteration:
    print("Page 8 not found in the document analysis.")

## Step 3: Minimal Vector Search

We will use OpenAI embeddings to convert these chunks into vectors and store them in a simple in-memory list. We'll then use cosine similarity to find the "best" chunk for a query.

In [ ]:
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import numpy as np

# Initialize OpenAI Client
# We use Entra ID (RBAC) because Key-based authentication is disabled on this resource
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
)

oai_client = AzureOpenAI(
    azure_endpoint=env["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=token_provider,
    api_version="2023-05-15"
)

embedding_model = env.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large") # Use env var or default

def get_embedding(text):
    response = oai_client.embeddings.create(input=[text], model=embedding_model)
    return response.data[0].embedding

# Embed a subset of chunks (to save time/cost in this lab, or all if small)
# For the full user guide (~100 pages), let's limit to first 200 chunks for the demo
demo_chunks = chunks[:200] 
print(f"Embedding {len(demo_chunks)} chunks...")

embeddings = [get_embedding(chunk) for chunk in demo_chunks]
print("Embeddings created.")

def naive_search(query, k=3):
    query_vec = get_embedding(query)
    similarities = [np.dot(query_vec, doc_vec) for doc_vec in embeddings]
    # Get top k indices
    top_k_indices = np.argsort(similarities)[-k:][::-1]
    return [(demo_chunks[i], similarities[i]) for i in top_k_indices]

print("Search ready.")

## Step 4: Failure Mode Analysis

Now let's ask questions that require understanding **tables** or **figures**.

### Test 1: Table Data
Attempt to retrieve specific specs that likely live in a table (e.g., "What is the weight of the device?"). Notice how the table row might be split across chunks, or the headers are missing from the chunk containing the value.

In [ ]:
# --- 🔍 EVIDENCE OF FAILURE 2: DATA INTEGRITY (TABLE DESTRUCTION) ---
from IPython.display import Image, display

# We will focus on the INDEX TABLE on Page 5 to show how structure is lost.
found_table = None
target_page = 5

try:
    # Find the specific table on Page 5 (The Index)
    found_table = next(t for t in result.tables if t.bounding_regions[0].page_number == target_page)
    
    print(f"--- 1. THE VISUAL REALITY (Index Table on Page {target_page}) ---")
    print("This is the structured data explicitly extracted by Azure AI Document Intelligence.")
    print("Notice the row/column relationships imply 'Topic' -> 'Page Number'.\n")
    
    # Display the visual reference
    try:
        display(Image(filename="page5.png", width=600))
    except:
        print("Image 'page5.png' not found. Please ensure it is in the module folder.")

    # Reconstruct the table for display
    print("\n[Ground Truth Table Reconstructed from Document Intelligence Data]:")
    # Create a grid
    grid = [["" for _ in range(found_table.column_count)] for _ in range(found_table.row_count)]
    for cell in found_table.cells:
        # Access cell content directly
        grid[cell.row_index][cell.column_index] = cell.content.replace("\n", " ")
    
    # Print as a nice ASCII table
    col_widths = [max(len(cell) for cell in col) + 2 for col in zip(*grid)]
    
    def print_row(row):
        print("|" + "|".join(cell.ljust(width) for cell, width in zip(row, col_widths)) + "|")

    print("-" * (sum(col_widths) + len(col_widths) + 1))
    for row in grid:
        print_row(row)
        print("-" * (sum(col_widths) + len(col_widths) + 1))
        
    print(f"\nThis table contains {len(found_table.cells)} cells of informative data.")

    print("\n--- 2. THE NAIVE CHUNKING REALITY (What the LLM sees) ---")
    print("Now we find the chunk that contains this text. Notice how the columns vanish.")
    
    # Find the chunks that contain this table's content
    # We search for "Concept of Circuit" + "7-8" which appear in the table
    # but likely appear 'mashed' in the chunk
    
    unique_phrase = "Concept of Circuit"
    
    matching_chunks = [c for c in chunks if unique_phrase in c]
    
    if matching_chunks:
        print(f"Found {len(matching_chunks)} chunk(s) containing the Index:\n")
        for i, c in enumerate(matching_chunks):
            print(f">>> CHUNK {i+1} PREVIEW:")
            print(c) 
            print("...\n")
            print("👉 CRITICAL FAILURE:")
            print("   1. Where is the 'Page No' column header?")
            print("   2. Is '7-8' clearly associated with 'Concept of Circuit' as a page number?")
            print("   3. Or does it look like 'Concept of Circuit and Network 7-8' is just one long sentence?")
            print("   4. Without structure, an LLM likely ignores the page numbers or thinks they are part of the title.")
    else:
        print("Could not locate the specific table text in the chunks.")

except StopIteration:
    print(f"Table on Page {target_page} not found.")

### Test 2: Figures and Diagrams (Page 12)
Technical documents rely heavily on diagrams. On Page 12, there is a circuit diagram.
Visual information is completely lost in naive text extraction.

**Discussion Question:** If you feed the text below to GPT-4, will it be able to draw the circuit? Will it know that R1 and R2 are in parallel?

In [ ]:
# --- 🔍 EVIDENCE OF FAILURE 3: FIGURE LOSS ---
from IPython.display import Image, display

target_page = 12

print(f"--- 1. THE VISUAL REALITY (Circuit Diagram on Page {target_page}) ---")
print("This defines the circuit topology (R1, R2, V, I directions).")
print("Crucially, the KCL equation 'Is = I1 + I2' only makes sense with this diagram.\n")

try:
    display(Image(filename="page12.png", width=600))
except:
    print("page12.png not found")

print("\n--- 2. THE NAIVE CHUNKING REALITY (What the LLM sees) ---")
print("We extract text from Page 12. Does it describe the connections?")
print("Does it say 'R1 is in parallel with R2'? Or just list labels?\n")

try:
    # Get page 12 text
    # Note: result.pages is a list, we find the one with page_number == 12
    p12 = next(p for p in result.pages if p.page_number == target_page)
    
    page_12_text = ""
    for span in p12.spans:
        page_12_text += result.content[span.offset : span.offset + span.length]
        
    print(">>> EXTRACTED TEXT FROM PAGE 12 (Partial):")
    # Show the text around where the diagram likely is (usually has labels like R1, R2)
    # or just print the whole page text if it's not too long
    print(page_12_text) 
    print("\n--------------------------")
    print("👉 CRITICAL FAILURE:")
    print("   1. The diagram is GONE.")
    print("   2. You might see labels like 'R1', 'R2', 'Vs' floating in the text, or mashed together.")
    print("   3. An LLM cannot reconstruct the circuit topology (Series vs Parallel) from this flat text.")

except StopIteration:
    print(f"Page {target_page} not found.")

## Conclusion

You have just seen **Naive RAG** in action. It is fast to build, but it destroys the meaning of complex documents.
In the next module, we will learn how to use **Document Intelligence** properly to preserve this structure.